## Подготовка `docs/tonihuy_faq.txt` для FAISS / RAG

Ниже — разбиение HTML на чанки в 2 этапа:

- **Этап 1 (семантика)**: извлекаем смысловые блоки из HTML (заголовки, абзацы, пункты списков, цитаты), игнорируя картинки/фигуры.
- **Этап 2 (размер)**: если блок слишком длинный — дробим на под-чанки по целевому размеру с overlap.

Выход: список `chunks` (каждый — `dict` с `text` и `metadata`) + сохранение в `tonihuy_faq_chunks.jsonl`.

In [10]:
from __future__ import annotations

import json
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Tuple

SRC_PATH = Path("docs/tonihuy_faq.txt")
OUT_PATH = Path("tonihuy_faq_chunks.jsonl")

raw_html = SRC_PATH.read_text(encoding="utf-8")
len(raw_html), raw_html[:200]

(22272,
 '<h3 dir="auto" id="Важные-ссылки-для-обучения">Важные ссылки для обучения</h3><p dir="auto">Самое нужное - <a href="https://applicant.21-school.ru/" target="_blank">applicant</a>, <a href="https://pla')

In [ ]:
def _collapse_ws(s: str) -> str:
    s = s.replace("\u00a0", " ")
    s = re.sub(r"[ \t\f\v]+", " ", s)
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()


def _html_to_blocks_bs4(html: str) -> List[Tuple[str, str]]:
    """Return list of (kind, text) blocks preserving reading order."""
    from bs4 import BeautifulSoup  # type: ignore

    soup = BeautifulSoup(html, "html.parser")

    # Убираем то, что почти всегда шумит в RAG
    for tag in soup.find_all(["script", "style", "noscript"]):
        tag.decompose()

    blocks: List[Tuple[str, str]] = []

    for el in soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6", "p", "li", "blockquote"]):
        # Пропускаем содержимое фигур/картинок
        if el.find_parent(["figure"]):
            continue

        # 1) Сохраняем ссылки: <a> -> "текст (url)"
        #    (так URL останется в чанке и сможет вернуться в ответе)
        for a in el.find_all("a"):
            href = (a.get("href") or "").strip()
            label = _collapse_ws(a.get_text(" ", strip=True))
            if href and label:
                a.replace_with(f"{label} ({href})")
            elif href:
                a.replace_with(f"({href})")
            else:
                # если href нет — оставляем только текст
                a.replace_with(label)

        # 2) Оставляем переносы только там, где они реально есть в HTML (<br>)
        for br in el.find_all("br"):
            br.replace_with("\n")

        text = el.get_text(" ", strip=True)
        text = _collapse_ws(text)
        if not text:
            continue

        # Служебные якоря телеграфа
        if text.lower() == "наверх" or "_tl_editor" in text:
            continue

        kind = el.name
        if kind == "li":
            text = f"- {text}"
        blocks.append((kind, text))

    return blocks


def _html_to_blocks_fallback(html: str) -> List[Tuple[str, str]]:
    """Fallback без зависимостей: грубо превращает HTML в блоки."""

    # Удаляем фигуры целиком (обычно это картинки + подписи)
    html = re.sub(r"<figure[\s\S]*?</figure>", "\n", html, flags=re.IGNORECASE)

    # <br> -> переносы
    html = re.sub(r"<br\s*/?>", "\n", html, flags=re.IGNORECASE)

    # Заголовки/параграфы/ли/blockquote как отдельные строки
    def repl(tag: str, prefix: str = ""):
        return rf"\n{prefix}\\1\n"

    # h1-h6
    html = re.sub(r"<h[1-6][^>]*>([\s\S]*?)</h[1-6]>", repl("h"), html, flags=re.IGNORECASE)
    # p
    html = re.sub(r"<p[^>]*>([\s\S]*?)</p>", repl("p"), html, flags=re.IGNORECASE)
    # li
    html = re.sub(r"<li[^>]*>([\s\S]*?)</li>", repl("li", prefix="- "), html, flags=re.IGNORECASE)
    # blockquote
    html = re.sub(r"<blockquote[^>]*>([\s\S]*?)</blockquote>", repl("blockquote", prefix="> "), html, flags=re.IGNORECASE)

    # Ссылки: оставляем текст + URL в скобках
    html = re.sub(r"<a\s+[^>]*href=\"([^\"]+)\"[^>]*>([\s\S]*?)</a>", r"\2 (\1)", html, flags=re.IGNORECASE)

    # Удаляем все оставшиеся теги
    html = re.sub(r"<[^>]+>", "", html)

    lines = [_collapse_ws(x) for x in html.splitlines()]
    lines = [x for x in lines if x]

    blocks: List[Tuple[str, str]] = []
    for ln in lines:
        # Служебные якоря телеграфа
        if ln.lower() == "наверх" or "_tl_editor" in ln:
            continue

        kind = "text"
        if ln.startswith("- "):
            kind = "li"
        elif ln.startswith("> "):
            kind = "blockquote"
        blocks.append((kind, ln))

    return blocks


def html_to_blocks(html: str) -> List[Tuple[str, str]]:
    try:
        import bs4  # type: ignore  # noqa: F401

        return _html_to_blocks_bs4(html)
    except Exception:
        return _html_to_blocks_fallback(html)


blocks = html_to_blocks(raw_html)
len(blocks), blocks[:10]

(87,
 [('h3', 'Важные ссылки для обучения'),
  ('p',
   'Самое нужное - applicant (https://applicant.21-school.ru/) , платформа (https://platform.21-school.ru/) и Rocket.Chat (https://rocketchat-student.21-school.ru/) : На аппликанте твои документы об обучении, генерация QR, информация о дедлайнах, магазин мерча за коины и заявки на стажировки. На платформе - создание мероприятий, проведение проверок оффлайн/онлайн, все проекты основы. В рокете - связь с разными кампусами, тематичесикие чаты (random coffee, ft_memes и пр., их можно найти в поиске), чаты волонтеров и тестирование новых фич. Из необязательных - чат основы в телеграме, отдельные чаты по проектам/веткам.'),
  ('p', 'Ряд полезных ссылок:'),
  ('li',
   '- Оферта (https://applicant.21-school.ru/contract) (содержит много важной инфы в т.ч. о заморозке и продлении ДДЛ)'),
  ('li', '- Правила Школы (https://applicant.21-school.ru/rules)'),
  ('li', '- Правила Рокетчата (https://applicant.21-school.ru/rocketchat)'),
  ('li', '- 

In [ ]:
@dataclass
class SemanticBlock:
    section: str
    text: str


def blocks_to_semantic_blocks(blocks: List[Tuple[str, str]]) -> List[SemanticBlock]:
    """Склеивает элементы до следующего заголовка.

    Идея: каждый chunk в RAG должен иметь понятный "якорь" (заголовок) + контент.
    """

    out: List[SemanticBlock] = []
    current_section = ""
    buf: List[str] = []

    def flush():
        nonlocal buf
        text = _collapse_ws("\n".join(buf))
        if text:
            out.append(SemanticBlock(section=current_section.strip() or "(no heading)", text=text))
        buf = []

    for kind, text in blocks:
        if kind in {"h1", "h2", "h3", "h4", "h5", "h6"}:
            flush()
            current_section = text
            # чтобы заголовок присутствовал внутри текста чанка (лучше для retrieval)
            buf.append(text)
        else:
            buf.append(text)

    flush()
    return out


semantic_blocks = blocks_to_semantic_blocks(blocks)
len(semantic_blocks), semantic_blocks[0].section, semantic_blocks[0].text[:300]

(16,
 'Важные ссылки для обучения',
 'Важные ссылки для обучения\nСамое нужное - applicant (https://applicant.21-school.ru/) , платформа (https://platform.21-school.ru/) и Rocket.Chat (https://rocketchat-student.21-school.ru/) : На аппликанте твои документы об обучении, генерация QR, информация о дедлайнах, магазин мерча за коины и заявк')

In [ ]:
def split_text_words(text: str, target_words: int = 220, overlap_words: int = 40) -> List[str]:
    """Простое дробление по словам (стабильно, без зависимостей)."""
    words = text.split()
    if len(words) <= target_words:
        return [text]

    chunks: List[str] = []
    i = 0
    step = max(1, target_words - overlap_words)
    while i < len(words):
        j = min(len(words), i + target_words)
        chunk = " ".join(words[i:j]).strip()
        if chunk:
            chunks.append(chunk)
        if j >= len(words):
            break
        i += step

    return chunks


def semantic_blocks_to_chunks(
    semantic_blocks: List[SemanticBlock],
    *,
    target_words: int = 240,
    overlap_words: int = 40,
    source: str = str(SRC_PATH),
) -> List[Dict[str, Any]]:
    chunks: List[Dict[str, Any]] = []
    chunk_id = 0

    for b in semantic_blocks:
        parts = split_text_words(b.text, target_words=target_words, overlap_words=overlap_words)
        for part_idx, part in enumerate(parts):
            chunk_id += 1
            chunks.append(
                {
                    "id": f"tonihuy_faq_{chunk_id:05d}",
                    "text": part,
                    "metadata": {
                        "source": source,
                        "section": b.section,
                        "part": part_idx,
                        "parts": len(parts),
                    },
                }
            )

    return chunks


chunks = semantic_blocks_to_chunks(semantic_blocks, target_words=240, overlap_words=40)
len(chunks), chunks[0]["metadata"], chunks[0]["text"][:220]

(18,
 {'source': 'docs/tonihuy_faq.txt',
  'section': 'Важные ссылки для обучения',
  'part': 0,
  'parts': 1},
 'Важные ссылки для обучения\nСамое нужное - applicant (https://applicant.21-school.ru/) , платформа (https://platform.21-school.ru/) и Rocket.Chat (https://rocketchat-student.21-school.ru/) : На аппликанте твои документы о')

In [ ]:
def write_jsonl(path: Path, rows: Iterable[Dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


write_jsonl(OUT_PATH, chunks)
OUT_PATH.resolve(), OUT_PATH.stat().st_size

(PosixPath('/Users/staspog/S21/S21_agent/tonihuy_faq_chunks.jsonl'), 30915)

In [ ]:
# Быстрый sanity-check: распределение размеров чанков (в словах)
lengths = [len(c["text"].split()) for c in chunks]
min(lengths), sum(lengths) / len(lengths), max(lengths), lengths[:10]

(39, 123.83333333333333, 240, [148, 139, 133, 50, 115, 92, 101, 86, 185, 118])

In [ ]:
# Посмотрим несколько первых чанков целиком
for c in chunks[:3]:
    print("=" * 80)
    print(c["id"], c["metadata"]["section"], f"(part {c['metadata']['part']+1}/{c['metadata']['parts']})")
    print(c["text"])
    print()

tonihuy_faq_00001 Важные ссылки для обучения (part 1/1)
Важные ссылки для обучения
Самое нужное - applicant (https://applicant.21-school.ru/) , платформа (https://platform.21-school.ru/) и Rocket.Chat (https://rocketchat-student.21-school.ru/) : На аппликанте твои документы об обучении, генерация QR, информация о дедлайнах, магазин мерча за коины и заявки на стажировки. На платформе - создание мероприятий, проведение проверок оффлайн/онлайн, все проекты основы. В рокете - связь с разными кампусами, тематичесикие чаты (random coffee, ft_memes и пр., их можно найти в поиске), чаты волонтеров и тестирование новых фич. Из необязательных - чат основы в телеграме, отдельные чаты по проектам/веткам.
Ряд полезных ссылок:
- Оферта (https://applicant.21-school.ru/contract) (содержит много важной инфы в т.ч. о заморозке и продлении ДДЛ)
- Правила Школы (https://applicant.21-school.ru/rules)
- Правила Рокетчата (https://applicant.21-school.ru/rocketchat)
- Гайд по выпуску (https://applicant.21-sch

## Обработка `docs/adm_faq.html`

Ниже — тот же пайплайн (полезный текст + ссылки), но для FAQ со страницы applicant (`adm_faq.html`).

In [ ]:
SRC_PATH_ADM = Path("docs/adm_faq.html")
OUT_PATH_ADM = Path("adm_faq_chunks.jsonl")

raw_html_adm = SRC_PATH_ADM.read_text(encoding="utf-8")
len(raw_html_adm), raw_html_adm[:200]

(50218,
 '<!DOCTYPE html>\n<!-- saved from url=(0034)https://applicant.21-school.ru/faq -->\n<html lang="ru"><head><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta name="viewport" content=')

In [ ]:
blocks_adm = html_to_blocks(raw_html_adm)
semantic_blocks_adm = blocks_to_semantic_blocks(blocks_adm)

chunks_adm = semantic_blocks_to_chunks(
    semantic_blocks_adm,
    target_words=240,
    overlap_words=40,
    source=str(SRC_PATH_ADM),
)

write_jsonl(OUT_PATH_ADM, chunks_adm)
len(blocks_adm), len(semantic_blocks_adm), len(chunks_adm), OUT_PATH_ADM.resolve()

(95, 8, 12, PosixPath('/Users/staspog/S21/S21_agent/adm_faq_chunks.jsonl'))

In [ ]:
# Sanity-check: убедимся, что ссылки реально попали в текст
with_links = [c for c in chunks_adm if "http" in c["text"]]
len(with_links), (with_links[0]["text"][:400] if with_links else "no links found")

(4,
 'Движение по графу a) Какие этапы мне нужно пройти, чтобы выпуститься? С октября 2025 предоставлена возможность проходить обучение по следующим направлениям: Разработка: - Backend - Frontend - Mobile Android - Mobile IOS - Gamedev - Разработчик С++ Неразработческие направления: - Data Science - BSA - Cybersecurity - DevOps - QA - Project Manager - UX/UI дизайн - Биоинформатика На старте обучения ес')

## Обработка `docs/adm_page.html`

Страница `ADM` (полезные ссылки по кампусам/правилам и т.п.). Обрабатываем тем же пайплайном: текст + ссылки → семантические секции → чанки → JSONL.

In [20]:
SRC_PATH_ADM_PAGE = Path("docs/adm_page.html")
OUT_PATH_ADM_PAGE = Path("adm_page_chunks.jsonl")

raw_html_adm_page = SRC_PATH_ADM_PAGE.read_text(encoding="utf-8")
len(raw_html_adm_page), raw_html_adm_page[:200]

(44836,
 '<!DOCTYPE html>\n<!-- saved from url=(0034)https://applicant.21-school.ru/adm -->\n<html lang="ru"><head><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta name="viewport" content=')

In [21]:
blocks_adm_page = html_to_blocks(raw_html_adm_page)
semantic_blocks_adm_page = blocks_to_semantic_blocks(blocks_adm_page)

chunks_adm_page = semantic_blocks_to_chunks(
    semantic_blocks_adm_page,
    target_words=240,
    overlap_words=40,
    source=str(SRC_PATH_ADM_PAGE),
)

write_jsonl(OUT_PATH_ADM_PAGE, chunks_adm_page)
len(blocks_adm_page), len(semantic_blocks_adm_page), len(chunks_adm_page), OUT_PATH_ADM_PAGE.resolve()

(77, 6, 9, PosixPath('/Users/staspog/S21/S21_agent/adm_page_chunks.jsonl'))

In [22]:
# Sanity-check: ссылки должны быть в тексте
with_links_page = [c for c in chunks_adm_page if "http" in c["text"]]
len(with_links_page), (with_links_page[0]["text"][:400] if with_links_page else "no links found")

(7,
 'в Сургуте на 2 этаже в кластере Hydrogen в ADM Support (стеклянная дверь) - в Великом Новгороде в кластере Quark - в Якутске на 2 этаже в кластере Erchim в Aquarium (стеклянная дверь) - в Ярославле стекляшка на 7 этаже - в Магасе на 2 этаже Азкабан (рядом с Орденом Феникса) - в Белгороде на 2 этаже (стеклянная дверь зеленого цвета) - в Южно-Сахалинске в кластере Taranai - в Челябинске на 4 этаже в')

## FAISS: эмбеддинги + индекс для 3 документов

Шаги:

- Загружаем чанки из:
  - `docs/tonihuy_faq_chunks.jsonl`
  - `docs/adm_page_chunks.jsonl`
  - `docs/adm_faq_chunks.jsonl`
- Векторизуем компактной CPU-моделью `intfloat/multilingual-e5-small` (нормализуем → cosine similarity).
- Строим FAISS индекс (`IndexFlatIP`).
- Сохраняем на диск: `faiss_store/index.faiss` + `faiss_store/chunks.jsonl` + `faiss_store/config.json`.

In [23]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

CHUNK_FILES = [
    Path("docs/tonihuy_faq_chunks.jsonl"),
    Path("docs/adm_page_chunks.jsonl"),
    Path("docs/adm_faq_chunks.jsonl"),
]

STORE_DIR = Path("faiss_store")
INDEX_PATH = STORE_DIR / "index.faiss"
CHUNKS_PATH = STORE_DIR / "chunks.jsonl"
CONFIG_PATH = STORE_DIR / "config.json"

MODEL_NAME = "intfloat/multilingual-e5-small"  # компактный, хорошо для RU на CPU

STORE_DIR.mkdir(parents=True, exist_ok=True)

# Проверим, что файлы на месте
[(p.as_posix(), p.exists(), p.stat().st_size if p.exists() else None) for p in CHUNK_FILES]

[('docs/tonihuy_faq_chunks.jsonl', True, 30915),
 ('docs/adm_page_chunks.jsonl', True, 15361),
 ('docs/adm_faq_chunks.jsonl', True, 21280)]

In [24]:
def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    return rows


all_chunks: List[Dict[str, Any]] = []
for p in CHUNK_FILES:
    rows = read_jsonl(p)
    # добавим информацию, из какого файла эти чанки
    for r in rows:
        r.setdefault("metadata", {})
        r["metadata"]["chunk_file"] = p.as_posix()
    all_chunks.extend(rows)

len(all_chunks), all_chunks[0].keys(), all_chunks[0]["metadata"]

(39,
 dict_keys(['id', 'text', 'metadata']),
 {'source': 'docs/tonihuy_faq.txt',
  'section': 'Важные ссылки для обучения',
  'part': 0,
  'parts': 1,
  'chunk_file': 'docs/tonihuy_faq_chunks.jsonl'})

In [25]:
# Готовим тексты для E5: passage/query префиксы важны для качества
passages = [f"passage: {c['text']}" for c in all_chunks]

model = SentenceTransformer(MODEL_NAME, device="cpu")

# encode возвращает np.ndarray float32
emb = model.encode(
    passages,
    batch_size=64,
    convert_to_numpy=True,
    normalize_embeddings=True,  # cosine -> inner product
    show_progress_bar=True,
)
emb = emb.astype(np.float32)
emb.shape, emb.dtype

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

((39, 384), dtype('float32'))

In [26]:
# Строим индекс
faiss.normalize_L2(emb)  # на всякий случай (если вдруг модель вернула не идеально нормализованные)

d = emb.shape[1]
index = faiss.IndexFlatIP(d)
index.add(emb)

index.ntotal, d

(39, 384)

In [27]:
# Сохраняем индекс и метаданные (тексты нужны для генерации ответа)
faiss.write_index(index, str(INDEX_PATH))
write_jsonl(CHUNKS_PATH, all_chunks)

config = {
    "model": MODEL_NAME,
    "index_type": "IndexFlatIP",
    "dim": int(d),
    "normalize_embeddings": True,
    "chunk_files": [p.as_posix() for p in CHUNK_FILES],
    "count": int(index.ntotal),
}
CONFIG_PATH.write_text(json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")

INDEX_PATH.resolve(), CHUNKS_PATH.resolve(), CONFIG_PATH.resolve()

(PosixPath('/Users/staspog/S21/S21_agent/faiss_store/index.faiss'),
 PosixPath('/Users/staspog/S21/S21_agent/faiss_store/chunks.jsonl'),
 PosixPath('/Users/staspog/S21/S21_agent/faiss_store/config.json'))

In [ ]:
# Быстрый тест поиска

def search(query: str, k: int = 5):
    q_emb = model.encode(
        [f"query: {query}"],
        convert_to_numpy=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0].tolist(), idxs[0].tolist()):
        c = all_chunks[idx]
        results.append(
            {
                "score": float(score),
                "id": c.get("id"),
                "section": c.get("metadata", {}).get("section"),
                "source": c.get("metadata", {}).get("source"),
                "text": c.get("text"),
            }
        )
    return results

search("где найти правила рокетчата", k=5)

[{'score': 0.8434760570526123,
  'id': 'tonihuy_faq_00001',
  'section': 'Важные ссылки для обучения',
  'source': 'docs/tonihuy_faq.txt',
  'text': 'Важные ссылки для обучения\nСамое нужное - applicant (https://applicant.21-school.ru/) , платформа (https://platform.21-school.ru/) и Rocket.Chat (https://rocketchat-student.21-school.ru/) : На аппликанте твои документы об обучении, генерация QR, информация о дедлайнах, магазин мерча за коины и заявки на стажировки. На платформе - создание мероприятий, проведение проверок оффлайн/онлайн, все проекты основы. В рокете - связь с разными кампусами, тематичесикие чаты (random coffee, ft_memes и пр., их можно найти в поиске), чаты волонтеров и тестирование новых фич. Из необязательных - чат основы в телеграме, отдельные чаты по проектам/веткам.\nРяд полезных ссылок:\n- Оферта (https://applicant.21-school.ru/contract) (содержит много важной инфы в т.ч. о заморозке и продлении ДДЛ)\n- Правила Школы (https://applicant.21-school.ru/rules)\n- Правил

In [30]:
print("\n".join(['stas', 'pog', 'pog']))

stas
pog
pog
